In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------------------
# Configuração geral
# ---------------------------------------------------------------------------
DATA_DIR = Path(".")
OUTPUT_DIR = Path("./graficos")
OUTPUT_DIR.mkdir(exist_ok=True)

YEARS = ["2023", "2024", "2025"]

COLOR_MASC = "#2a78d6"
COLOR_FEM = "#eb6834"

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10


def load_data():
    return {y: pd.read_csv(DATA_DIR / f"{y}.csv", low_memory=False) for y in YEARS}


def working(df):
    """Filtra apenas quem tem cargo preenchido (está de fato trabalhando com dados)."""
    return df[df["cargo_atual"].notna()]


def save(fig, name):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{name}.png", bbox_inches="tight")
    plt.close(fig)
    print(f"salvo: {OUTPUT_DIR / name}.png")


# ---------------------------------------------------------------------------
# Faixa salarial -> ponto médio numérico (proxy, pois a pesquisa coleta faixas)
# ---------------------------------------------------------------------------
SALARY_MIDPOINTS = {
    "Menos de R$ 1.000/mês": 750,
    "de R$ 1.001/mês a R$ 2.000/mês": 1500,
    "de R$ 101/mês a R$ 2.000/mês": 1500,
    "de R$ 2.001/mês a R$ 3.000/mês": 2500,
    "de R$ 3.001/mês a R$ 4.000/mês": 3500,
    "de R$ 4.001/mês a R$ 6.000/mês": 5000,
    "de R$ 6.001/mês a R$ 8.000/mês": 7000,
    "de R$ 8.001/mês a R$ 12.000/mês": 10000,
    "de R$ 12.001/mês a R$ 16.000/mês": 14000,
    "de R$ 16.001/mês a R$ 20.000/mês": 18000,
    "de R$ 20.001/mês a R$ 25.000/mês": 22500,
    "de R$ 25.001/mês a R$ 30.000/mês": 27500,
    "de R$ 25.001/mês a R$ 3000/mês": 27500,
    "de R$ 30.001/mês a R$ 40.000/mês": 35000,
    "Acima de R$ 40.001/mês": 42500,
}

NIVEIS_ORDEM = ["Júnior", "Pleno", "Sênior", "Especialista/Staff+"]

# Rótulos de cargo variam ligeiramente entre anos no dataset original —
# por isso o mapeamento é feito ano a ano.
CARGO_LABELS = {
    "2023": {
        "Analista de Dados": "Analista de Dados/Data Analyst",
        "Cientista de Dados": "Cientista de Dados/Data Scientist",
        "Engenheiro de Dados": "Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect",
        "Analista de BI": "Analista de BI/BI Analyst",
        "Analista de Negócios": "Analista de Negócios/Business Analyst",
        "Dev/Eng. Software": "Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas",
    },
    "2024": {
        "Analista de Dados": "Analista de Dados/Data Analyst",
        "Cientista de Dados": "Cientista de Dados/Data Scientist",
        "Engenheiro de Dados": "Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect",
        "Analista de BI": "Analista de BI/BI Analyst",
        "Analista de Negócios": "Analista de Negócios/Business Analyst",
        "Analytics Engineer": "Analytics Engineer",
    },
    "2025": {
        "Analista de Dados": "Analista de Dados/Data Analyst",
        "Cientista de Dados": "Cientista de Dados/Data Scientist",
        "Engenheiro de Dados": "Engenheiro de Dados/Data Engineer/Data Architect",
        "Analista de BI": "Analista de BI/BI Analyst",
        "Analista de Negócios": "Analista de Negócios/Business Analyst",
        "Analytics Engineer": "Analytics Engineer",
        "ML/AI Engineer": "Engenheiro de Machine Learning/ML Engineer/AI Engineer",
    },
}

SETOR_LABELS = {
    "Tecnologia": "Tecnologia/Fábrica de Software",
    "Finanças/Bancos": "Finanças ou Bancos",
    "Varejo": "Varejo",
    "Consultoria": "Área de Consultoria",
    "Indústria": "Indústria",
    "Saúde": "Área da Saúde",
}

MODELO_TRABALHO_LABELS = {
    "100% remoto": "Modelo 100% remoto",
    "Híbrido flexível": "Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)",
    "Híbrido dias fixos": "Modelo híbrido com dias fixos de trabalho presencial",
    "100% presencial": "Modelo 100% presencial",
}


# ---------------------------------------------------------------------------
# 1. VISÃO GERAL DO MERCADO — cargos (por ano, % e quantidade), senioridade, setores
# ---------------------------------------------------------------------------
def secao_1_visao_geral(dfs):
    # --- cargos mais comuns por ano: % e quantidade (6 gráficos) ---
    for y in YEARS:
        dft = working(dfs[y])
        vc = dft["cargo_atual"].value_counts()
        vc = vc.drop(labels=["Outra Opção"], errors="ignore")
        top = vc.head(7 if y == "2025" else 6)
        labels = [CARGO_LABELS[y].get(  # tenta achar nome curto correspondente
            next((k for k, v in CARGO_LABELS[y].items() if v == raw), raw), raw)
            for raw in top.index]
        # nome curto: se o raw bater com algum valor do dicionário do ano, usa a chave
        labels = []
        for raw in top.index:
            curto = next((k for k, v in CARGO_LABELS[y].items() if v == raw), raw)
            labels.append(curto)

        # gráfico de quantidade
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.bar(labels, top.values, color="#2a78d6")
        ax.set_title(f"Cargos mais comuns — {y} (quantidade)")
        ax.set_ylabel("Número de respondentes")
        plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
        save(fig, f"01a_cargos_{y}_quantidade")

        # gráfico de %
        pct = (top / len(dft) * 100).round(1)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.bar(labels, pct.values, color="#2a78d6")
        ax.set_title(f"Cargos mais comuns — {y} (%)")
        ax.set_ylabel("% dos respondentes")
        plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
        save(fig, f"01b_cargos_{y}_percentual")

    # --- evolução dos principais cargos em % (linha), usando rótulo certo por ano ---
    cargos_evolucao = ["Analista de Dados", "Cientista de Dados", "Engenheiro de Dados",
                        "Analista de BI", "Analista de Negócios", "Analytics Engineer"]
    fig, ax = plt.subplots(figsize=(8, 5))
    for cargo in cargos_evolucao:
        valores = []
        for y in YEARS:
            dft = working(dfs[y])
            pct = dft["cargo_atual"].value_counts(normalize=True) * 100
            raw = CARGO_LABELS[y].get(cargo)
            valores.append(round(pct.get(raw, np.nan), 1) if raw else np.nan)
        ax.plot(YEARS, valores, marker="o", label=cargo)
    ax.set_title("Evolução dos principais cargos — 2023 a 2025 (%)")
    ax.set_ylabel("% dos respondentes")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    save(fig, "02_cargos_evolucao_pct")

    # --- senioridade: evolução em % (linha) ---
    fig, ax = plt.subplots(figsize=(7, 5))
    for nivel in NIVEIS_ORDEM:
        valores = []
        for y in YEARS:
            dft = working(dfs[y])
            pct = dft["nivel_cargo_atual"].value_counts(normalize=True) * 100
            valores.append(round(pct.get(nivel, 0), 1))
        ax.plot(YEARS, valores, marker="o", label=nivel)
    ax.set_title("Evolução da senioridade — 2023 a 2025")
    ax.set_ylabel("% dos respondentes")
    ax.legend()
    save(fig, "03_senioridade_evolucao")

    # --- setores: barras comparando os 3 anos (rótulos corretos) ---
    x = np.arange(len(SETOR_LABELS))
    width = 0.25
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, y in enumerate(YEARS):
        dft = working(dfs[y])
        pct = dft["setor_atuacao"].value_counts(normalize=True) * 100
        valores = [round(pct.get(raw, 0), 1) for raw in SETOR_LABELS.values()]
        ax.bar(x + i * width, valores, width, label=y)
    ax.set_xticks(x + width)
    ax.set_xticklabels(SETOR_LABELS.keys(), rotation=20, ha="right")
    ax.set_title("Top setores que empregam profissionais de dados")
    ax.set_ylabel("% dos respondentes")
    ax.legend(title="Ano")
    save(fig, "04_setores_comparacao")


# ---------------------------------------------------------------------------
# 2. PAINEL DE INDICADORES (grid) + MODELO DE TRABALHO 100% EMPILHADO
# ---------------------------------------------------------------------------
def secao_2_painel(dfs):
    linhas = []
    for y, df in dfs.items():
        total = len(df)
        dft = working(df)
        pct_trabalha = len(dft) / total * 100
        gestor = pd.to_numeric(df["flag_gestor"], errors="coerce").sum()
        pct_lideranca = gestor / len(dft) * 100
        linhas.append({"Indicador": y, "Nº de respondentes": f"{total:,}".replace(",", "."),
                        "% trabalhando com dados": f"{pct_trabalha:.1f}%",
                        "% em posição de liderança": f"{pct_lideranca:.1f}%"})
    painel = pd.DataFrame(linhas).set_index("Indicador").T
    print("\n--- Painel de indicadores ---")
    print(painel)

    # grid visual do painel (tabela renderizada como imagem)
    fig, ax = plt.subplots(figsize=(7, 2.2))
    ax.axis("off")
    tabela = ax.table(cellText=painel.values, rowLabels=painel.index,
                       colLabels=painel.columns, cellLoc="center", loc="center")
    tabela.auto_set_font_size(False)
    tabela.set_fontsize(11)
    tabela.scale(1, 2.2)
    for (row, col), cell in tabela.get_celld().items():
        if row == 0:
            cell.set_facecolor("#f2ede4")
            cell.set_text_props(weight="bold")
    ax.set_title("Painel de indicadores — 2023 a 2025", pad=20, fontsize=13, weight="bold")
    save(fig, "05_painel_indicadores")

    # modelo de trabalho — só existe em 2023 e 2025
    modelo_pct = {}
    for y in ["2023", "2025"]:
        dft = working(dfs[y])
        vc = dft["modelo_atual_de_trabalho"].value_counts()
        tot = vc.sum()
        remoto = vc.get(MODELO_TRABALHO_LABELS["100% remoto"], 0)
        presencial = vc.get(MODELO_TRABALHO_LABELS["100% presencial"], 0)
        hibrido = tot - remoto - presencial
        modelo_pct[y] = {"Remoto": round(remoto / tot * 100, 1),
                          "Híbrido": round(hibrido / tot * 100, 1),
                          "Presencial": round(presencial / tot * 100, 1)}

    anos_modelo = list(modelo_pct.keys())
    remoto = [modelo_pct[y]["Remoto"] for y in anos_modelo]
    hibrido = [modelo_pct[y]["Híbrido"] for y in anos_modelo]
    presencial = [modelo_pct[y]["Presencial"] for y in anos_modelo]

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.barh(anos_modelo, remoto, color="#2a78d6", label="Remoto")
    ax.barh(anos_modelo, hibrido, left=remoto, color="#1baf7a", label="Híbrido")
    left2 = [r + h for r, h in zip(remoto, hibrido)]
    ax.barh(anos_modelo, presencial, left=left2, color="#eb6834", label="Presencial")
    ax.set_xlim(0, 100)
    ax.set_xlabel("%")
    ax.set_title("Modelo de trabalho 100% empilhado — 2023 vs 2025\n(2024 não tem essa pergunta na pesquisa)")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    save(fig, "06_modelo_trabalho_empilhado")


# ---------------------------------------------------------------------------
# 3. DIVERSIDADE DE GÊNERO
# ---------------------------------------------------------------------------
def secao_3_genero(dfs):
    # evolução % masculino x feminino
    masc, fem = [], []
    for y in YEARS:
        vc = dfs[y]["genero"].value_counts(normalize=True) * 100
        masc.append(round(vc.get("Masculino", 0), 1))
        fem.append(round(vc.get("Feminino", 0), 1))

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(YEARS, masc, marker="o", color=COLOR_MASC, label="Masculino")
    ax.plot(YEARS, fem, marker="o", color=COLOR_FEM, label="Feminino")
    ax.set_title("Evolução de gênero no mercado de dados")
    ax.set_ylabel("% dos respondentes")
    ax.legend()
    save(fig, "07_genero_evolucao")

    # composição de gênero por cargo (2025)
    cargos_curto = ["Analista de Dados", "Cientista de Dados", "Engenheiro de Dados",
                     "Analista de BI", "Analista de Negócios"]
    cargos_raw_2025 = [CARGO_LABELS["2025"][c] for c in cargos_curto]
    dft = working(dfs["2025"])
    sub = dft[dft["cargo_atual"].isin(cargos_raw_2025) & dft["genero"].isin(["Masculino", "Feminino"])]
    tab = pd.crosstab(sub["cargo_atual"], sub["genero"], normalize="index").reindex(cargos_raw_2025) * 100

    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(cargos_curto))
    ax.bar(x, tab["Masculino"].values, color=COLOR_MASC, label="Masculino")
    ax.bar(x, tab["Feminino"].values, bottom=tab["Masculino"].values, color=COLOR_FEM, label="Feminino")
    ax.set_xticks(x)
    ax.set_xticklabels(cargos_curto, rotation=15, ha="right")
    ax.set_ylabel("% dentro do cargo")
    ax.set_title("Composição de gênero por cargo — 2025")
    ax.legend()
    save(fig, "08_genero_por_cargo_2025")

    # salário mediano por gênero e senioridade — 2025 (todos os níveis)
    d = dfs["2025"].copy()
    d = d[d["genero"].isin(["Masculino", "Feminino"])]
    d["sal_num"] = d["faixa_salarial"].map(SALARY_MIDPOINTS)
    tab_sal = d.groupby(["nivel_cargo_atual", "genero"])["sal_num"].median().unstack().reindex(NIVEIS_ORDEM)

    x = np.arange(len(NIVEIS_ORDEM))
    width = 0.35
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(x - width / 2, tab_sal["Masculino"], width, color=COLOR_MASC, label="Masculino")
    ax.bar(x + width / 2, tab_sal["Feminino"], width, color=COLOR_FEM, label="Feminino")
    ax.set_xticks(x)
    ax.set_xticklabels(NIVEIS_ORDEM)
    ax.set_ylabel("R$ (ponto médio da faixa)")
    ax.set_title("Salário mediano por gênero e senioridade — 2025")
    ax.legend()
    save(fig, "09_salario_mediano_genero_senioridade_2025")

    # salário mediano de Sênior por gênero — evolução 2023 a 2025
    salario_masc, salario_fem = [], []
    for y in YEARS:
        d = dfs[y].copy()
        d = d[d["genero"].isin(["Masculino", "Feminino"])]
        d["sal_num"] = d["faixa_salarial"].map(SALARY_MIDPOINTS)
        med = d[d["nivel_cargo_atual"] == "Sênior"].groupby("genero")["sal_num"].median()
        salario_masc.append(med.get("Masculino", np.nan))
        salario_fem.append(med.get("Feminino", np.nan))

    x = np.arange(len(YEARS))
    width = 0.35
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(x - width / 2, salario_masc, width, color=COLOR_MASC, label="Masculino")
    ax.bar(x + width / 2, salario_fem, width, color=COLOR_FEM, label="Feminino")
    ax.set_xticks(x)
    ax.set_xticklabels(YEARS)
    ax.set_ylabel("R$ (ponto médio da faixa)")
    ax.set_title("Salário mediano de Sênior por gênero — 2023 a 2025")
    ax.legend()
    save(fig, "10_salario_mediano_genero_senior_evolucao")

    # heatmaps: cargo x gênero(% fem) x ano  |  nível x gênero(% fem) x ano
    def heatmap_pct_fem_cargo():
        cargos_heatmap = ["Analista de Dados", "Cientista de Dados", "Engenheiro de Dados", "Analista de BI"]
        matriz = []
        for cargo in cargos_heatmap:
            linha = []
            for y in YEARS:
                dft = working(dfs[y])
                raw = CARGO_LABELS[y].get(cargo)
                if raw is None:
                    linha.append(np.nan)
                    continue
                sub = dft[dft["cargo_atual"].isin(CARGO_LABELS[y].values()) & dft["genero"].isin(["Masculino", "Feminino"])]
                tab = pd.crosstab(sub["cargo_atual"], sub["genero"], normalize="index") * 100
                linha.append(round(tab["Feminino"].get(raw, np.nan), 1))
            matriz.append(linha)
        matriz = pd.DataFrame(matriz, index=cargos_heatmap, columns=YEARS)

        fig, ax = plt.subplots(figsize=(7, 4.5))
        sns.heatmap(matriz, annot=True, fmt=".1f", cmap="Oranges", cbar_kws={"label": "% feminino"},
                    linewidths=1, linecolor="white", ax=ax)
        ax.set_title("% de mulheres por cargo e ano")
        save(fig, "11_heatmap_genero_cargo_ano")

    def heatmap_pct_fem_nivel():
        matriz = []
        for nivel in NIVEIS_ORDEM:
            linha = []
            for y in YEARS:
                dft = working(dfs[y])
                sub = dft[dft["nivel_cargo_atual"].isin(NIVEIS_ORDEM) & dft["genero"].isin(["Masculino", "Feminino"])]
                tab = pd.crosstab(sub["nivel_cargo_atual"], sub["genero"], normalize="index") * 100
                linha.append(round(tab["Feminino"].get(nivel, np.nan), 1) if nivel in tab.index else np.nan)
            matriz.append(linha)
        matriz = pd.DataFrame(matriz, index=NIVEIS_ORDEM, columns=YEARS)

        fig, ax = plt.subplots(figsize=(7, 4.5))
        sns.heatmap(matriz, annot=True, fmt=".1f", cmap="Oranges", cbar_kws={"label": "% feminino"},
                    linewidths=1, linecolor="white", ax=ax)
        ax.set_title("% de mulheres por nível de senioridade e ano")
        save(fig, "12_heatmap_genero_nivel_ano")

    heatmap_pct_fem_cargo()
    heatmap_pct_fem_nivel()


# ---------------------------------------------------------------------------
# 4. TECNOLOGIAS — linguagens, cloud, BI
# ---------------------------------------------------------------------------
def secao_4_tecnologias(dfs):
    linguagens = {"SQL": [], "Python": [], "R": []}
    for y in ["2023", "2024"]:
        dft = working(dfs[y])
        for lang, col in [("Python", "flag_utiliza_python"), ("SQL", "flag_utiliza_sql"), ("R", "flag_utiliza_r")]:
            respondidos = dft[col].dropna()
            pct = pd.to_numeric(respondidos, errors="coerce").astype(bool).mean() * 100
            linguagens[lang].append(round(pct, 1))
    dft = working(dfs["2025"])
    respondidos = dft["linguagem_favorita"].dropna()
    for lang in ["Python", "SQL", "R"]:
        pct = respondidos.str.contains(rf"\b{lang}\b", case=False, regex=True, na=False).mean() * 100
        linguagens[lang].append(round(pct, 1))

    fig, ax = plt.subplots(figsize=(6, 4))
    for lang, valores in linguagens.items():
        ax.plot(YEARS, valores, marker="o", label=lang)
    ax.set_title("Adoção de linguagens — 2023 a 2025")
    ax.set_ylabel("% de quem trabalha com dados")
    ax.legend()
    save(fig, "13_linguagens_evolucao")

    col_map = {
        "2023": {"AWS": "flag_utiliza_aws", "Azure": "flag_utiliza_azure", "GCP": "flag_utiliza_google_cloud"},
        "2024": {"AWS": "flag_utiliza_aws", "Azure": "flag_utiliza_azure", "GCP": "flag_utiliza_google_cloud"},
        "2025": {"AWS": "flag_amazon_web_services_aws", "Azure": "flag_azure_microsoft", "GCP": "flag_google_cloud_gcp"},
    }
    cloud = {"AWS": [], "Azure": [], "GCP": []}
    for y in YEARS:
        dft = working(dfs[y])
        for tech, col in col_map[y].items():
            respondidos = dft[col].dropna()
            pct = pd.to_numeric(respondidos, errors="coerce").astype(bool).mean() * 100
            cloud[tech].append(round(pct, 1))

    fig, ax = plt.subplots(figsize=(6, 4))
    for tech, valores in cloud.items():
        ax.plot(YEARS, valores, marker="o", label=tech)
    ax.set_title("Adoção de plataformas cloud — 2023 a 2025")
    ax.set_ylabel("% de quem trabalha com dados")
    ax.legend()
    save(fig, "14_cloud_evolucao")

    bi_col_map = {
        "2023": {"Power BI": None, "Tableau": "flag_utiliza_tableau", "Looker": "flag_utiliza_looker"},
        "2024": {"Power BI": "flag_microsoft_powerbi", "Tableau": "flag_tableau", "Looker": "flag_looker"},
        "2025": {"Power BI": "falg_utiliza_power_bi", "Tableau": "flag_utiliza_sql_tableau", "Looker": "flag_looker"},
    }
    bi = {"Power BI": [], "Tableau": [], "Looker": []}
    for y in YEARS:
        dft = working(dfs[y])
        for tool, col in bi_col_map[y].items():
            if tool == "Power BI" and y == "2023":
                texto = dft["flag_utiliza_powerbi"].dropna()
                pct = texto.str.contains("PowerBI").mean() * 100
            else:
                respondidos = dft[col].dropna()
                pct = pd.to_numeric(respondidos, errors="coerce").astype(bool).mean() * 100
            bi[tool].append(round(pct, 1))

    fig, ax = plt.subplots(figsize=(6, 4))
    for tool, valores in bi.items():
        ax.plot(YEARS, valores, marker="o", label=tool)
    ax.set_title("Adoção de ferramentas de BI — 2023 a 2025\n(base de respondentes bem menor em 2023)")
    ax.set_ylabel("% de quem respondeu a pergunta de BI")
    ax.legend()
    save(fig, "15_bi_evolucao")


# ---------------------------------------------------------------------------
# 5. IA GENERATIVA (só existe a partir de 2024)
# ---------------------------------------------------------------------------
def secao_5_ia(dfs):
    # prioridade estratégica da empresa
    categorias = ["Principal prioridade", "Entre principais (2-4 anos)",
                  "Iniciativa isolada, pouco foco", "Não é prioridade"]
    # Nota: os rótulos exatos das opções de resposta variam ano a ano no dataset
    # original; os valores abaixo foram apurados manualmente a partir das colunas
    # 'ai_generativa_e_uma_prioridade_em_sua_empresa' (2024) e
    # 'ai_generativa_e_llm_e_uma_prioridade' (2025).
    valores_2024 = [12.3, 23.9, 30.7, 28.8]
    valores_2025 = [23.8, 36.8, 25.9, 11.3]

    x = np.arange(len(categorias))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width / 2, valores_2024, width, label="2024")
    ax.bar(x + width / 2, valores_2025, width, label="2025")
    ax.set_xticks(x)
    ax.set_xticklabels(categorias, rotation=15, ha="right")
    ax.set_ylabel("% das empresas")
    ax.set_title("IA generativa como prioridade da empresa — 2024 vs 2025")
    ax.legend()
    save(fig, "16_ia_prioridade")

    # uso pessoal de IA no trabalho
    col_ia = {"2024": "utiliza_chatgpt_ou_llms_no_trabalho2", "2025": "usa_chatgpt_ou_copilot_no_trabalho"}
    naotexto = "Não utilizo nenhum tipo de solução de IA Generativa"
    uso_pessoal = []
    for y in ["2024", "2025"]:
        dft = working(dfs[y])
        respondidos = dft[col_ia[y]].notna().sum()
        nao_usa = dft[col_ia[y]].fillna("").str.contains(naotexto).sum()
        usa = respondidos - nao_usa
        uso_pessoal.append(round(usa / respondidos * 100, 1))

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(["2024", "2025"], uso_pessoal, color="#2a78d6")
    ax.set_ylim(0, 100)
    ax.set_ylabel("% dos profissionais")
    ax.set_title("Uso pessoal de ChatGPT/Copilot/LLMs no trabalho")
    save(fig, "17_ia_uso_pessoal")

    # a empresa está conseguindo bons resultados com LLMs? (só 2025, pergunta nova)
    categorias_resultado = ["Sim, em produção\ngerando resultado", "Em parte, só pilotos\nsem muito impacto",
                             "Não, só investigação\n/planejamento", "Não começou\nnenhum projeto", "Não sei opinar"]
    valores_resultado = [27.5, 39.9, 15.6, 14.0, 6.9]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(categorias_resultado, valores_resultado, color="#2a78d6")
    ax.set_ylabel("% das empresas")
    ax.set_title("A empresa está conseguindo bons resultados com LLMs? — 2025")
    save(fig, "18_ia_resultados_2025")

    # principais barreiras para uso de IA generativa (2024 vs 2025)
    barreiras = ["Falta compreensão\nde casos de uso", "Falta expertise\ntécnica",
                 "Preocupação com\nsegurança/privacidade", "Dados não\nestão prontos",
                 "Baixa qualidade\n/alucinação", "ROI não\ncomprovado"]
    barreiras_2024 = [36.9, 34.8, 31.9, 30.7, 15.7, 15.3]
    barreiras_2025 = [32.0, 38.8, 27.8, 36.8, 17.0, 28.4]

    x = np.arange(len(barreiras))
    width = 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - width / 2, barreiras_2024, width, label="2024")
    ax.bar(x + width / 2, barreiras_2025, width, label="2025")
    ax.set_xticks(x)
    ax.set_xticklabels(barreiras)
    ax.set_ylabel("% das empresas (multi-escolha)")
    ax.set_title("Principais barreiras para uso de IA generativa nas empresas")
    ax.legend()
    save(fig, "19_ia_barreiras")


# ---------------------------------------------------------------------------
# 6. REGIÃO, SENIORIDADE E MODELO DE TRABALHO
# ---------------------------------------------------------------------------
def secao_6_regiao_senioridade(dfs):
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-oeste", "Norte"]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for regiao in regioes:
        valores = []
        for y in YEARS:
            dft = working(dfs[y])
            pct = dft["regiao_residencia"].value_counts(normalize=True) * 100
            valores.append(round(pct.get(regiao, 0), 1))
        ax.plot(YEARS, valores, marker="o", label=regiao)
    ax.set_title("Distribuição regional de profissionais de dados")
    ax.set_ylabel("% dos profissionais")
    ax.legend()
    save(fig, "20_regiao_distribuicao")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for regiao in regioes:
        valores = []
        for y in YEARS:
            d = dfs[y].copy()
            d["sal_num"] = d["faixa_salarial"].map(SALARY_MIDPOINTS)
            media = d[d["regiao_residencia"] == regiao]["sal_num"].mean()
            valores.append(round(media, 0))
        ax.plot(YEARS, valores, marker="o", label=regiao)
    ax.set_title("Salário médio por região — 2023 a 2025")
    ax.set_ylabel("R$ (ponto médio da faixa)")
    ax.legend()
    save(fig, "21_regiao_salario")

    fig, ax = plt.subplots(figsize=(6, 4))
    for nivel in NIVEIS_ORDEM:
        valores = []
        for y in YEARS:
            d = dfs[y].copy()
            d["sal_num"] = d["faixa_salarial"].map(SALARY_MIDPOINTS)
            med = d[d["nivel_cargo_atual"] == nivel]["sal_num"].median()
            valores.append(med)
        ax.plot(YEARS, valores, marker="o", label=nivel)
    ax.set_title("Salário mediano por senioridade — 2023 a 2025")
    ax.set_ylabel("R$ (ponto médio da faixa)")
    ax.legend()
    save(fig, "22_senioridade_salario")

    # salário médio por modelo de trabalho, apenas Sênior — 2023 vs 2025 (rótulos corretos)
    x = np.arange(len(MODELO_TRABALHO_LABELS))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, y in enumerate(["2023", "2025"]):
        d = dfs[y].copy()
        d["sal_num"] = d["faixa_salarial"].map(SALARY_MIDPOINTS)
        sub = d[d["nivel_cargo_atual"] == "Sênior"]
        valores = [sub[sub["modelo_atual_de_trabalho"] == raw]["sal_num"].mean()
                   for raw in MODELO_TRABALHO_LABELS.values()]
        ax.bar(x + i * width, valores, width, label=y)
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(MODELO_TRABALHO_LABELS.keys(), rotation=15, ha="right")
    ax.set_ylabel("R$ (ponto médio da faixa)")
    ax.set_title("Salário médio por modelo de trabalho — apenas nível Sênior")
    ax.legend(title="Ano")
    save(fig, "23_modelo_trabalho_salario")


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    dfs = load_data()
    secao_1_visao_geral(dfs)
    secao_2_painel(dfs)
    secao_3_genero(dfs)
    secao_4_tecnologias(dfs)
    secao_5_ia(dfs)
    secao_6_regiao_senioridade(dfs)
    print(f"\nTodos os gráficos foram salvos em: {OUTPUT_DIR.resolve()}")

salvo: graficos/01a_cargos_2023_quantidade.png
salvo: graficos/01b_cargos_2023_percentual.png
salvo: graficos/01a_cargos_2024_quantidade.png
salvo: graficos/01b_cargos_2024_percentual.png
salvo: graficos/01a_cargos_2025_quantidade.png
salvo: graficos/01b_cargos_2025_percentual.png
salvo: graficos/02_cargos_evolucao_pct.png
salvo: graficos/03_senioridade_evolucao.png
salvo: graficos/04_setores_comparacao.png

--- Painel de indicadores ---
Indicador                   2023   2024   2025
Nº de respondentes         8.542  5.293  3.495
% trabalhando com dados    69.8%  72.9%  71.6%
% em posição de liderança  23.9%  23.2%  29.1%
salvo: graficos/05_painel_indicadores.png
salvo: graficos/06_modelo_trabalho_empilhado.png
salvo: graficos/07_genero_evolucao.png
salvo: graficos/08_genero_por_cargo_2025.png
salvo: graficos/09_salario_mediano_genero_senioridade_2025.png
salvo: graficos/10_salario_mediano_genero_senior_evolucao.png
salvo: graficos/11_heatmap_genero_cargo_ano.png
salvo: graficos/12_hea